<a href="https://colab.research.google.com/github/casper-justus/swahili-gpt/blob/main/MiniGPT_Kiswahili_Resumable_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇰🇪 Swahili GPT — Multi-Day Resumable Training

A ~**30 Million parameter** GPT-style language model trained from scratch on a pure Kiswahili text corpus.
Built with **JAX**, **Flax (NNX API)**, and **Optax**. Designed to run on Google Colab's free T4 GPU across multiple sessions.

---

## 📋 Before You Start (Read This First!)

### ✅ Step 1: Enable the GPU
> Go to **Runtime → Change runtime type → T4 GPU → Save**.
> Without a GPU, each training step will be ~100x slower and the notebook will likely time out.

### ✅ Step 2: Upload your Tokenizer
> This notebook requires a pre-trained BPE tokenizer file called `kenya_tokenizer.json`.
> Upload it to your **Google Drive** inside the folder:
> `/MyDrive/MiniGPT_Kiswahili_Checkpoints/kenya_tokenizer.json`
> *(Storing it in Drive means it survives session restarts without re-uploading.)*

### ✅ Step 3: Always run cells top to bottom
> **Every time Colab disconnects and you reconnect**, you must run ALL cells from Cell 1 downward.
> Cell 7 will automatically detect your last saved checkpoint and resume from where you left off.

---

## 🧠 Model Architecture
| Hyperparameter | Value |
|---|---|
| Parameters | ~30 Million |
| Context Window | 1024 tokens (~750 Swahili words) |
| Vocabulary Size | 10,000 tokens (custom BPE) |
| Embedding Dim | 512 |
| Attention Heads | 8 |
| Transformer Blocks | 6 |
| Training Target | 200,000 steps (Chinchilla-optimal) |
| Batch Size | 4 (fits 16GB T4 VRAM) |

---

## 🗃️ Multi-Day Training Strategy
Google Colab's free tier disconnects sessions after **~12 hours**. This notebook handles that automatically:
- **Every 5,000 steps**, the model and optimizer state are saved to your Google Drive.
- When you reconnect, Cell 7 detects the latest checkpoint and resumes from that exact step.
- At ~4,096 tokens/step on a T4 GPU, **200,000 steps takes approximately 8–12 hours** of compute.
- You will likely need **1–2 Colab sessions** to complete the full run.

---

## 📦 Dataset
Uses [`marcoharuni95/swahili-text-corpus`](https://huggingface.co/datasets/marcoharuni95/swahili-text-corpus) — a modern, deduplicated Parquet-format Kiswahili corpus loaded directly from Hugging Face.


## Cell 1 — Mount Google Drive
Mounts your personal Google Drive so checkpoints and the tokenizer are available across all sessions.
When prompted, click the link, sign into your Google account, and paste the authorization code back here.


In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = "/content/drive/MyDrive/MiniGPT_Kiswahili_Checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"✅ Google Drive mounted. Checkpoints stored at: {CKPT_DIR}")


## Cell 2 — Install Dependencies
Uninstalls conflicting pre-installed JAX/Flax versions and installs pinned CUDA-12-compatible packages.
This cell takes **2–3 minutes**. The output will be long — that is completely normal.
> ⚠️ **Do not skip this cell.** Colab ships with JAX/Flax versions that are often incompatible with each other.


In [ ]:
# Cell 2: Install pinned CUDA-12 compatible libraries
!pip uninstall -y jax jaxlib flax optax orbax-checkpoint
!pip install "jax[cuda12]" flax optax orbax-checkpoint datasets tokenizers


## Cell 3 — Imports & Hyperparameters
All model hyperparameters are defined here. This config is specifically tuned to:
- Fit within **16GB VRAM** of the free T4 GPU (`BATCH_SIZE=4` with 1024 context).
- Hit the **Chinchilla-optimal** ratio of ~20 tokens per parameter at 200,000 steps.

> 💡 **CUDA Out of Memory?** Reduce `BATCH_SIZE` to `2`.
> 💡 **Want a faster test run?** Set `TOTAL_STEPS = 5000` to verify the pipeline works before committing to a full run.


In [ ]:
# Cell 3: Imports and Hyperparameters
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import orbax.checkpoint as ocp
from datasets import load_dataset
from tokenizers import Tokenizer

print(f"JAX version : {jax.__version__}")
print(f"Devices     : {jax.devices()}")

SEQ_LEN       = 1024    # Context window: tokens the model reads at once
BATCH_SIZE    = 4       # Kept at 4 to prevent OOM with 1024 context on T4
EMB_SIZE      = 512     # Token embedding vector size
NUM_HEADS     = 8       # Parallel self-attention heads per block
NUM_LAYERS    = 6       # Stacked Transformer blocks (network depth)
LEARNING_RATE = 3e-4    # AdamW learning rate
TOTAL_STEPS   = 200000  # Chinchilla-optimal: ~800M tokens for 30M params
SAVE_EVERY    = 5000    # Save checkpoint to Drive every 5,000 steps


## Cell 4 — Load Dataset & Tokenizer
Loads `kenya_tokenizer.json` from Google Drive and the Kiswahili text corpus from Hugging Face.
Encodes all text into a flat list of integer token IDs for efficient batching.

> ⚠️ This cell takes **3–5 minutes** on first run due to tokenizing ~17 million tokens.
> On subsequent reconnects, it runs again from scratch — this is expected.


In [ ]:
# Cell 4: Load Tokenizer and Kiswahili Dataset
print("Loading Tokenizer...")
tokenizer  = Tokenizer.from_file(f"{CKPT_DIR}/kenya_tokenizer.json")
VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"✅ Tokenizer loaded! Vocabulary size: {VOCAB_SIZE}")

print("\nLoading Kiswahili Dataset from Hugging Face...")
dataset = load_dataset("marcoharuni95/swahili-text-corpus", split="train")
print(f"✅ Dataset loaded! Total examples: {len(dataset)}")

def create_data_generator(dataset, tokenizer, batch_size, seq_len):
    all_tokens = []
    print("\nTokenizing data (this may take a few minutes)...")
    for text in dataset['text'][:200000]:
        if text:
            all_tokens.extend(tokenizer.encode(text).ids)
    print(f"✅ Total tokens: {len(all_tokens):,}")
    print(f"   Enough for ~{len(all_tokens) // (batch_size * seq_len):,} steps before looping.")
    i = 0
    while True:
        batch_x, batch_y = [], []
        for _ in range(batch_size):
            if i + seq_len + 1 >= len(all_tokens):
                i = 0
            chunk = all_tokens[i : i + seq_len + 1]
            batch_x.append(chunk[:-1])
            batch_y.append(chunk[1:])
            i += seq_len
        yield jnp.array(batch_x), jnp.array(batch_y)

dataloader = create_data_generator(dataset, tokenizer, BATCH_SIZE, SEQ_LEN)


## Cell 5 — Model Architecture
Defines the GPT-style Transformer with two important design decisions:
- **Pre-LayerNorm:** `LayerNorm` is applied *before* attention and MLP blocks for more stable gradients.
- **`nnx.Sequential` wrapper:** Blocks are wrapped in `nnx.Sequential` (not a plain Python list) to register them as a valid JAX Pytree, preventing `TypeError` crashes during JIT compilation.


In [ ]:
# Cell 5: GPT-Style Architecture
class Block(nnx.Module):
    def __init__(self, emb_size, num_heads, rngs):
        self.ln_1 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.attn = nnx.MultiHeadAttention(num_heads=num_heads, in_features=emb_size, decode=False, rngs=rngs)
        self.ln_2 = nnx.LayerNorm(emb_size, rngs=rngs)
        self.mlp  = nnx.Sequential(
            nnx.Linear(emb_size, 4 * emb_size, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * emb_size, emb_size, rngs=rngs)
        )

    def __call__(self, x, mask):
        x = x + self.attn(self.ln_1(x), mask=mask)  # Residual attention
        x = x + self.mlp(self.ln_2(x))              # Residual MLP
        return x

class MiniGPT(nnx.Module):
    def __init__(self, vocab_size, seq_len, emb_size, num_heads, num_layers, rngs):
        self.token_emb = nnx.Embed(vocab_size, emb_size, rngs=rngs)
        self.pos_emb   = nnx.Embed(seq_len, emb_size, rngs=rngs)
        # nnx.Sequential required here to prevent Pytree/JIT errors with plain Python lists
        self.blocks    = nnx.Sequential(*[Block(emb_size, num_heads, rngs) for _ in range(num_layers)])
        self.ln_f      = nnx.LayerNorm(emb_size, rngs=rngs)
        self.lm_head   = nnx.Linear(emb_size, vocab_size, rngs=rngs)

    def __call__(self, idx):
        b, t = idx.shape
        pos  = jnp.arange(0, t, dtype=jnp.int32)[None, :]
        x    = self.token_emb(idx) + self.pos_emb(pos)
        mask = nnx.make_causal_mask(jnp.ones((b, t)))
        for block in self.blocks.layers:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))

print("✅ Architecture defined.")


## Cell 6 — Initialize Model & Optimizer
Initializes the model and the AdamW optimizer with two Flax 0.11+ compatibility fixes:
1. **`wrt=nnx.Param`** — Tells the optimizer to only update trainable parameters.
2. **`optimizer.update(model, grads)`** — Both model and gradients must be passed explicitly (changed in Flax 0.11).

> ⚠️ The **first `train_step` call will take 60–90 seconds** as JAX JIT-compiles the entire graph for the GPU. All subsequent steps will be near-instant.


In [ ]:
# Cell 6: Initialize Model and Optimizer
rngs      = nnx.Rngs(0)
model     = MiniGPT(VOCAB_SIZE, SEQ_LEN, EMB_SIZE, NUM_HEADS, NUM_LAYERS, rngs)
tx        = optax.adamw(learning_rate=LEARNING_RATE)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)  # Fix 1: explicit wrt=nnx.Param

@nnx.jit
def train_step(model, optimizer, batch_x, batch_y):
    def loss_fn(model):
        logits = model(batch_x)
        return optax.softmax_cross_entropy_with_integer_labels(logits, batch_y).mean()
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)  # Fix 2: pass both model and grads (Flax 0.11+ requirement)
    return loss

print("✅ Model and optimizer initialized.")
print("   First train_step will take ~60-90 seconds for JIT compilation. This is normal!")


## Cell 7 — Resume / Restore Checkpoint
Scans your Google Drive folder for the latest saved checkpoint.
- **Found:** Restores the exact model weights AND AdamW optimizer momentum state. Training resumes seamlessly.
- **Not found:** Starts fresh from Step 0.

> ⚠️ **Key fix:** `nnx.split()` is called **separately** on `model` and `optimizer`. Passing both objects in one call causes a `TypeError: unsupported format string passed to Optimizer.__format__` in newer Flax versions.


In [ ]:
# Cell 7: Resume / Restore Checkpoint Logic
options = ocp.CheckpointManagerOptions(max_to_keep=3, create=True)
mngr    = ocp.CheckpointManager(CKPT_DIR, options=options)

# FIX: Split model and optimizer separately (prevents TypeError in Flax 0.11+)
_, model_state = nnx.split(model)
_, opt_state   = nnx.split(optimizer)
state_tree     = {'model': model_state, 'opt': opt_state}

start_step = 0

if mngr.latest_step() is not None:
    start_step = mngr.latest_step()
    print(f"✅ Found checkpoint! Resuming from step {start_step}...")
    restored = mngr.restore(start_step, args=ocp.args.StandardRestore(state_tree))
    nnx.update(model, restored['model'])
    nnx.update(optimizer, restored['opt'])
    print("✅ Model and Optimizer weights fully restored from Google Drive.")
else:
    print("ℹ️  No checkpoint found. Starting fresh from Step 0.")


## Cell 8 — Training Loop
Runs the main training loop from `start_step` to `TOTAL_STEPS`.
- Prints loss, speed (steps/sec), and ETA every **50 steps**.
- Saves full model + optimizer state to Google Drive every **5,000 steps**.

**Expected loss curve:**
| Step Range | Expected Loss | What the model is learning |
|---|---|---|
| 0 | ~9.2 | Random initialization |
| 0 – 5,000 | 9.2 → 5.0 | Token co-occurrence, spaces, punctuation |
| 5,000 – 50,000 | 5.0 → 3.5 | Common words, basic grammar patterns |
| 50,000 – 200,000 | 3.5 → 2.5 | Verb conjugation, noun class agreements |

> ✅ A final loss of **2.5–3.0** means the model is generating meaningful Kiswahili structure.


In [ ]:
# Cell 8: Training Loop
import time
print("🚀 Starting Training...")
print(f"   From step {start_step} / {TOTAL_STEPS} | Saving every {SAVE_EVERY} steps\n")

t0 = time.time()

for step in range(start_step, TOTAL_STEPS):
    batch_x, batch_y = next(dataloader)
    loss = train_step(model, optimizer, batch_x, batch_y)

    if step % 50 == 0:
        elapsed    = time.time() - t0
        steps_done = step - start_step + 1
        sps        = steps_done / elapsed if elapsed > 0 else 0
        eta_h      = ((TOTAL_STEPS - step) / sps) / 3600 if sps > 0 else float('inf')
        print(f"Step {step:06d} | Loss: {loss:.4f} | {sps:.2f} steps/s | ETA: {eta_h:.1f}h")

    if step > 0 and step % SAVE_EVERY == 0:
        # FIX: Split model and optimizer separately before saving
        _, current_model_state = nnx.split(model)
        _, current_opt_state   = nnx.split(optimizer)
        current_tree = {'model': current_model_state, 'opt': current_opt_state}
        mngr.save(step, args=ocp.args.StandardSave(current_tree))
        print(f"   💾 Checkpoint saved to Google Drive at step {step}")

print("\n🎉 Training complete!")
